# 2 · Parallel tool execution

When the model replies with several `tool_calls` in one assistant message, we
can run them independently. Tools that don't depend on each other are perfect
candidates for parallelism.

The executor exposes two strategies:
- **sequential** — run one at a time, in order;
- **parallel** — fan out onto worker threads and collect results back in the
  original order.

This demo is **fully local** (no network): we register `N` toy tools that each
sleep `300 ms`, then run the same batch both ways and compare the wall-clock
time. The ordering of results is preserved in *both* modes — parallelism buys
speed, never reordering.


In [ ]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::tools::{ToolRegistry, Tool, ToolResult, s_required};
use agent_loop::executor::execute_many;
use serde_json::{json, Value};
use std::time::Duration;

// Register 4 independent "sleep" tools. Each claims 300 ms of work.
let mut registry = ToolRegistry::new();
for i in 0..4usize {
    registry.register(Tool::new(
        format!("compute_{i}"),
        format!("Simulate compute task {i}"),
        s_required(json!({}), &[]),
        move |_| {
            std::thread::sleep(Duration::from_millis(300));
            Ok(ToolResult::ok(format!("task {i} done")))
        },
    ));
}

// The model "emitted" 4 tool calls in a single assistant message.
let calls: Vec<agent_loop::chat::ToolCall> = (0..4).map(|i| agent_loop::chat::ToolCall {
    id: format!("call_{i}"),
    name: format!("compute_{i}"),
    arguments: "{}".into(),
}).collect();
println!("model emitted {} tool calls in ONE message", calls.len());


### Sequential run

Every call waits for the previous one: 4 × 300 ms ≈ **1200 ms**.


In [ ]:

let (seq_results, seq_stats) = execute_many(&registry, &calls, false);
println!("sequential: {} calls took {:.1} ms", seq_stats.n_calls, seq_stats.elapsed_ms);
for r in &seq_results {
    println!("   ⇠ {}", r.content.as_deref().unwrap_or(""));
}


### Parallel run

Workers run concurrently: ≈ **300 ms** for the whole batch, ~4× faster.
Results are returned in the *same order* as the calls.


In [ ]:

let (par_results, par_stats) = execute_many(&registry, &calls, true);
println!("parallel  : {} calls took {:.1} ms", par_stats.n_calls, par_stats.elapsed_ms);
for r in &par_results {
    println!("   ⇠ {}", r.content.as_deref().unwrap_or(""));
}


### Side-by-side

```
sequential : 1200 ms   ████████████████████████████
parallel   :  300 ms   ██████
```

The speedup scales with the number of *independent* calls. The loop chooses the
strategy via `AgentConfig.parallel_tools`. The same `execute_many` is what the
agent loop in demos 4–5 calls under the hood.
